<a href="https://colab.research.google.com/github/Supan-Roy/financial-scam-analysis/blob/main/notebooks/08_synthetic_data_augmentation_evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Notebook 08: Synthetic Data Augmentation for Binary Financial Scam Detection

## 1. Research Objective

This notebook investigates whether the synthetic financial scam datasets previously developed and used in the multi-level classification experiments can improve binary financial scam detection.

The investigation reuses the established synthetic scam datasets:

- V3.1 synthetic financial scam dataset.
- LLM-authored financial scam dataset.

Both datasets were previously incorporated into the research pipeline for subtype classification, fraudulent intent analysis, social-engineering tactic recognition, and cross-generation evaluation.

This notebook extends their use to binary scam-versus-legitimate classification.

The experiments compare three training configurations using the same DistilBERT architecture:

- **Experiment A — Real-only:** Real financial scam + real legitimate communications.
- **Experiment B — Synthetic:** Previously established synthetic financial scam data + real legitimate communications.
- **Experiment C — Combined:** Real financial scam + previously established synthetic financial scam data + real legitimate communications.

Experiment A has already been completed in Notebook 06. Its trained model and benchmark results will be reused rather than retrained.

## 2. Experimental Sequence

1. **Existing-model evaluation:** Evaluate the saved real-only DistilBERT model on synthetic scam communications paired with real legitimate communications. No additional training or fine-tuning will be performed.
2. **Synthetic-only training:** Train a new model using the established synthetic scam datasets and real legitimate communications.
3. **Combined training:** Train a new model using real scam, synthetic scam, and real legitimate communications.
4. **Comparative evaluation:** Compare all three configurations on consistent real-world and synthetic evaluation distributions.

## 3. Dataset Preparation

The previously established raw and processed synthetic datasets will be inspected before use.

The inspection will verify:

- Dataset schemas and available text and label fields.
- Existing preprocessing and normalization conventions.
- Binary label compatibility.
- Record counts and dataset provenance.
- Duplicate and near-duplicate relationships within and across synthetic datasets.
- Compatibility with the preprocessing and tokenization pipeline used in Notebook 06.

Previously processed datasets will be reused where compatible, avoiding unnecessary repeated preprocessing.

## 4. Evaluation Strategy

Two evaluation distributions will be considered:

- **Real-world evaluation:** The existing untouched group-aware real test partition containing real scam and real legitimate communications.
- **Synthetic evaluation:** Held-out synthetic scam examples paired with real legitimate test examples.

The synthetic evaluation will be explicitly identified as a mixed-distribution evaluation rather than an independent real-world benchmark.

## 5. Experimental Controls

- Preserve the existing real train, validation, and test partitions.
- Independently partition synthetic data before training.
- Prevent duplicate or near-duplicate records from crossing synthetic training, validation, and test partitions.
- Maintain consistent model architecture, tokenizer, preprocessing conventions, and evaluation metrics.
- Compare natural-size and size-controlled training configurations where appropriate.
- Keep all test examples outside training and model selection.

## 6. Evaluation Metrics

- Accuracy
- Scam-class precision
- Scam-class recall
- Scam-class F1-score
- Macro F1-score
- Precision-Recall AUC
- Confusion matrix

## 7. Research Questions

1. Can the existing real-only model recognize previously unseen synthetic scam narratives?
2. Does synthetic scam training improve detection of real financial scams?
3. Does combining real and synthetic scam data improve binary classification?
4. Are improvements attributable to synthetic data augmentation or simply increased training volume?

## 8. Interpretation Principles

Synthetic evaluation performance will not be treated as evidence of real-world generalization by itself.

Results will be interpreted according to dataset composition, provenance, and evaluation distribution. All findings, including negative or inconclusive results, will be reported transparently.


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [7]:
# Verify the runtime environment, GPU, dataset files, and saved DistilBERT model

import os
import sys
import torch
from pathlib import Path

PROJECT_ROOT = Path("/content/financial-scam-analysis")
DRIVE_ROOT = Path("/content/drive/MyDrive/financial-scam-analysis")

REAL_SPLITS_DIR = DRIVE_ROOT / "data" / "splits"
SYNTHETIC_DATA_PATH = DRIVE_ROOT / "data" / "raw" / "synthetic_financial_scams.csv"
SAVED_MODEL_DIR = DRIVE_ROOT / "models" / "distilbert_binary"

print("Python version:", sys.version.split()[0])
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("\nProject root exists:", PROJECT_ROOT.exists())
print("Drive root exists:", DRIVE_ROOT.exists())

print("\nReal dataset partitions:")
for filename in ["train.csv", "validation.csv", "test.csv"]:
    path = REAL_SPLITS_DIR / filename
    print(f"{filename}: {'FOUND' if path.exists() else 'MISSING'}")

print("\nSynthetic dataset:")
print(SYNTHETIC_DATA_PATH)
print("Status:", "FOUND" if SYNTHETIC_DATA_PATH.exists() else "MISSING")

print("\nSaved DistilBERT model:")
print(SAVED_MODEL_DIR)
print("Directory exists:", SAVED_MODEL_DIR.exists())

if SAVED_MODEL_DIR.exists():
    required_files = [
        "config.json",
        "model.safetensors",
        "tokenizer.json",
        "tokenizer_config.json"
    ]

    for filename in required_files:
        path = SAVED_MODEL_DIR / filename
        print(f"{filename}: {'FOUND' if path.exists() else 'MISSING'}")

Python version: 3.13.16
PyTorch version: 2.14.1+cu130
CUDA available: True
GPU: Tesla T4

Project root exists: False
Drive root exists: True

Real dataset partitions:
train.csv: FOUND
validation.csv: FOUND
test.csv: FOUND

Synthetic dataset:
/content/drive/MyDrive/financial-scam-analysis/data/raw/synthetic_financial_scams.csv
Status: MISSING

Saved DistilBERT model:
/content/drive/MyDrive/financial-scam-analysis/models/distilbert_binary
Directory exists: True
config.json: FOUND
model.safetensors: FOUND
tokenizer.json: FOUND
tokenizer_config.json: FOUND


In [9]:
# Locate the synthetic financial scam dataset within the project directory

from pathlib import Path

DATA_ROOT = DRIVE_ROOT / "data"

print("Available data directories:\n")

for path in sorted(DATA_ROOT.rglob("*")):
    if path.is_file() and (
        "synthetic" in path.name.lower()
        or "scam" in path.name.lower()
    ):
        print(path)

Available data directories:

/content/drive/MyDrive/financial-scam-analysis/data/processed/llm_authored_financial_scams_processed.csv
/content/drive/MyDrive/financial-scam-analysis/data/processed/real_fin_scam_processed.csv
/content/drive/MyDrive/financial-scam-analysis/data/processed/synthetic_financial_scams_v3_1_processed.csv
/content/drive/MyDrive/financial-scam-analysis/data/raw/llm_authored_financial_scams.csv
/content/drive/MyDrive/financial-scam-analysis/data/raw/real_fin_scam.csv
/content/drive/MyDrive/financial-scam-analysis/data/raw/synthetic_financial_scams_v3_1.csv


In [10]:
# Inspect raw and processed synthetic datasets and compare them with the established real scam dataset

import pandas as pd

DATA_PATHS = {
    "Real Scam (Processed)": DRIVE_ROOT / "data/processed/real_fin_scam_processed.csv",
    "V3.1 Synthetic (Raw)": DRIVE_ROOT / "data/raw/synthetic_financial_scams_v3_1.csv",
    "V3.1 Synthetic (Processed)": DRIVE_ROOT / "data/processed/synthetic_financial_scams_v3_1_processed.csv",
    "LLM-authored (Raw)": DRIVE_ROOT / "data/raw/llm_authored_financial_scams.csv",
    "LLM-authored (Processed)": DRIVE_ROOT / "data/processed/llm_authored_financial_scams_processed.csv",
}

datasets = {}

for name, path in DATA_PATHS.items():
    if not path.exists():
        print(f"\nMISSING: {name} -> {path}")
        continue

    df = pd.read_csv(path, low_memory=False)
    datasets[name] = df

    print(f"\n{'=' * 80}")
    print(name)
    print("=" * 80)
    print("Shape:", df.shape)
    print("\nColumns:")
    print(df.columns.tolist())
    print("\nData types:")
    print(df.dtypes.to_string())

    for column in ["label", "scam_subtype", "fraudulent_intent", "social_engineering_tactics", "is_synthetic"]:
        if column in df.columns:
            print(f"\n{column} distribution:")
            print(df[column].value_counts(dropna=False).head(15).to_string())

    text_columns = [
        col for col in ["original_text", "normalized_text", "text", "message", "email_body"]
        if col in df.columns
    ]
    print("\nAvailable text columns:", text_columns)

    for column in text_columns:
        print(f"\nSample from {column}:")
        sample = df[column].dropna()
        if not sample.empty:
            print(str(sample.iloc[0])[:500])


Real Scam (Processed)
Shape: (13861, 30)

Columns:
['record_id', 'original_text', 'normalized_text', 'label', 'channel', 'original_channel', 'is_email_gateway', 'has_gateway_headers', 'source', 'source_url', 'source_license', 'source_record_id', 'is_synthetic', 'scam_subtype', 'fraudulent_intent', 'social_engineering_tactics', 'legitimate_category', 'email_subject', 'email_body', 'duplicate_cluster_id', 'char_count_original', 'word_count_original', 'char_count_normalized', 'word_count_normalized', 'placeholder_count_original', 'placeholder_count_normalized', 'generation_scenario', 'risk_or_safety_note', 'language', 'communication_type']

Data types:
record_id                        object
original_text                    object
normalized_text                  object
label                             int64
channel                          object
original_channel                 object
is_email_gateway                   bool
has_gateway_headers                bool
source               

In [11]:
# Audit existing synthetic partitions, duplicate clusters, and label consistency

for name in ["V3.1 Synthetic (Processed)", "LLM-authored (Processed)"]:
    df = datasets[name]

    print(f"\n{'=' * 75}")
    print(name)
    print("=" * 75)

    print("\nLabel values:")
    print(df["label"].value_counts(dropna=False).to_string())

    if "split_partition" in df.columns:
        print("\nExisting split_partition distribution:")
        print(df["split_partition"].value_counts(dropna=False).to_string())
    else:
        print("\nNo split_partition column found.")

    if "duplicate_cluster_id" in df.columns:
        cluster = df["duplicate_cluster_id"]
        print("\nDuplicate cluster audit:")
        print("Missing cluster IDs:", cluster.isna().sum())
        print("Unique non-null clusters:", cluster.nunique())
        print("Records in multi-record clusters:",
              cluster.value_counts().loc[lambda x: x > 1].sum()
              if cluster.notna().any() else 0)

    if "generation_scenario" in df.columns:
        print("\nGeneration scenario count:", df["generation_scenario"].nunique(dropna=True))

    if "scenario_family" in df.columns:
        print("Scenario family count:", df["scenario_family"].nunique(dropna=True))

    print("\nNormalized text missing:", df["normalized_text"].isna().sum())
    print("Normalized text empty:",
          df["normalized_text"].fillna("").str.strip().eq("").sum())

    print("Duplicate normalized texts:", df["normalized_text"].duplicated().sum())


V3.1 Synthetic (Processed)

Label values:
label
1    7800

Existing split_partition distribution:
split_partition
train    6240
val       780
test      780

Duplicate cluster audit:
Missing cluster IDs: 7627
Unique non-null clusters: 85
Records in multi-record clusters: 173

Generation scenario count: 104
Scenario family count: 104

Normalized text missing: 0
Normalized text empty: 0
Duplicate normalized texts: 43

LLM-authored (Processed)

Label values:
label
phishing    2730

Existing split_partition distribution:
split_partition
train    2184
val       273
test      273

Duplicate cluster audit:
Missing cluster IDs: 2730
Unique non-null clusters: 0
Records in multi-record clusters: 0
Scenario family count: 130

Normalized text missing: 0
Normalized text empty: 0
Duplicate normalized texts: 0


In [12]:
# Check whether duplicate records, duplicate clusters, or generation scenarios cross synthetic partitions

def audit_cross_partition_leakage(df, dataset_name):
    print(f"\n{'=' * 75}")
    print(dataset_name)
    print("=" * 75)

    for column in ["normalized_text", "duplicate_cluster_id", "generation_scenario", "scenario_family"]:
        if column not in df.columns:
            continue

        valid = df.dropna(subset=[column]).copy()

        if column == "normalized_text":
            valid = valid[valid[column].astype(str).str.strip().ne("")]

        group_partitions = valid.groupby(column)["split_partition"].nunique()
        crossing_groups = group_partitions[group_partitions > 1]

        print(f"\n{column}:")
        print("Unique groups:", valid[column].nunique())
        print("Groups appearing in multiple partitions:", len(crossing_groups))

        if len(crossing_groups):
            affected = valid[valid[column].isin(crossing_groups.index)]
            print("Affected records:", len(affected))
            print("Partition distribution of affected records:")
            print(affected["split_partition"].value_counts().to_string())
        else:
            print("No cross-partition overlap detected.")

for name in ["V3.1 Synthetic (Processed)", "LLM-authored (Processed)"]:
    audit_cross_partition_leakage(datasets[name], name)


V3.1 Synthetic (Processed)

normalized_text:
Unique groups: 7757
Groups appearing in multiple partitions: 0
No cross-partition overlap detected.

duplicate_cluster_id:
Unique groups: 85
Groups appearing in multiple partitions: 0
No cross-partition overlap detected.

generation_scenario:
Unique groups: 104
Groups appearing in multiple partitions: 0
No cross-partition overlap detected.

scenario_family:
Unique groups: 104
Groups appearing in multiple partitions: 0
No cross-partition overlap detected.

LLM-authored (Processed)

normalized_text:
Unique groups: 2730
Groups appearing in multiple partitions: 0
No cross-partition overlap detected.

duplicate_cluster_id:
Unique groups: 0
Groups appearing in multiple partitions: 0
No cross-partition overlap detected.

scenario_family:
Unique groups: 130
Groups appearing in multiple partitions: 0
No cross-partition overlap detected.


In [13]:
# Load the untouched real test partition and verify its legitimate examples

REAL_TEST_PATH = REAL_SPLITS_DIR / "test.csv"

real_test_df = pd.read_csv(REAL_TEST_PATH, low_memory=False)

print("Real test shape:", real_test_df.shape)
print("\nLabel distribution:")
print(real_test_df["label"].value_counts(dropna=False).sort_index())

print("\nSynthetic flag distribution:")
if "is_synthetic" in real_test_df.columns:
    print(real_test_df["is_synthetic"].value_counts(dropna=False))

print("\nRequired columns:")
required_columns = ["normalized_text", "label"]
for column in required_columns:
    print(f"{column}: {'FOUND' if column in real_test_df.columns else 'MISSING'}")

real_legit_test = real_test_df.loc[
    real_test_df["label"].eq(0)
].copy()

print("\nLegitimate test records:", len(real_legit_test))
print("Missing normalized text:", real_legit_test["normalized_text"].isna().sum())
print(
    "Empty normalized text:",
    real_legit_test["normalized_text"].fillna("").str.strip().eq("").sum()
)

print("\nSample legitimate message:")
print(real_legit_test["normalized_text"].iloc[0][:500])

Real test shape: (7618, 32)

Label distribution:
label
0    6233
1    1385
Name: count, dtype: int64

Synthetic flag distribution:
is_synthetic
False    7618
Name: count, dtype: int64

Required columns:
normalized_text: FOUND
label: FOUND

Legitimate test records: 6233
Missing normalized text: 0
Empty normalized text: 0

Sample legitimate message:
Your OTP is 789654, keep it confidential to secure your account.


In [14]:
# Construct separate synthetic evaluation datasets using the existing test partitions

v31_df = datasets["V3.1 Synthetic (Processed)"]
llm_df = datasets["LLM-authored (Processed)"]

v31_scam_test = v31_df.loc[
    v31_df["split_partition"].eq("test"),
    ["normalized_text"]
].copy()

v31_scam_test["label"] = 1
v31_scam_test["source"] = "V3.1 Synthetic"

llm_scam_test = llm_df.loc[
    llm_df["split_partition"].eq("test"),
    ["normalized_text"]
].copy()

llm_scam_test["label"] = 1
llm_scam_test["source"] = "LLM-authored"

legit_eval = real_legit_test[["normalized_text"]].copy()
legit_eval["label"] = 0
legit_eval["source"] = "Real Legitimate"

v31_eval_df = pd.concat(
    [v31_scam_test, legit_eval],
    ignore_index=True
)

llm_eval_df = pd.concat(
    [llm_scam_test, legit_eval],
    ignore_index=True
)

print("V3.1 evaluation set:", v31_eval_df.shape)
print(v31_eval_df["label"].value_counts().sort_index())
print("\nV3.1 source distribution:")
print(v31_eval_df["source"].value_counts())

print("\nLLM-authored evaluation set:", llm_eval_df.shape)
print(llm_eval_df["label"].value_counts().sort_index())
print("\nLLM-authored source distribution:")
print(llm_eval_df["source"].value_counts())

print("\nData integrity:")
for name, df in [
    ("V3.1", v31_eval_df),
    ("LLM-authored", llm_eval_df)
]:
    print(
        f"{name}: missing={df['normalized_text'].isna().sum()}, "
        f"empty={df['normalized_text'].fillna('').str.strip().eq('').sum()}"
    )

print(
    "\nIdentical legitimate evaluation examples:",
    v31_eval_df.loc[v31_eval_df["label"].eq(0), "normalized_text"].reset_index(drop=True).equals(
        llm_eval_df.loc[llm_eval_df["label"].eq(0), "normalized_text"].reset_index(drop=True)
    )
)

V3.1 evaluation set: (7013, 3)
label
0    6233
1     780
Name: count, dtype: int64

V3.1 source distribution:
source
Real Legitimate    6233
V3.1 Synthetic      780
Name: count, dtype: int64

LLM-authored evaluation set: (6506, 3)
label
0    6233
1     273
Name: count, dtype: int64

LLM-authored source distribution:
source
Real Legitimate    6233
LLM-authored        273
Name: count, dtype: int64

Data integrity:
V3.1: missing=0, empty=0
LLM-authored: missing=0, empty=0

Identical legitimate evaluation examples: True


In [15]:
# Load the previously trained DistilBERT binary classifier

from transformers import AutoTokenizer, AutoModelForSequenceClassification

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_path = str(SAVED_MODEL_DIR)

tokenizer = AutoTokenizer.from_pretrained(model_path)

model = AutoModelForSequenceClassification.from_pretrained(
    model_path
).to(device)

model.eval()

print("Model loaded successfully.")
print("Model:", model.config._name_or_path)
print("Device:", device)
print("Number of labels:", model.config.num_labels)
print("Label mapping:", model.config.id2label)
print("Maximum sequence length:", model.config.max_position_embeddings)
print("Model parameters:", sum(p.numel() for p in model.parameters()))
print("Model in evaluation mode:", not model.training)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Model loaded successfully.
Model: /content/drive/MyDrive/financial-scam-analysis/models/distilbert_binary
Device: cuda
Number of labels: 2
Label mapping: {0: 'LEGITIMATE', 1: 'SCAM'}
Maximum sequence length: 512
Model parameters: 66955010
Model in evaluation mode: True


In [16]:
# Run zero-shot inference on V3.1 and LLM-authored synthetic evaluation sets

import numpy as np
from tqdm.auto import tqdm

def run_inference(model, tokenizer, dataframe, batch_size=32, max_length=512):
    texts = dataframe["normalized_text"].tolist()
    all_probabilities = []

    model.eval()

    with torch.inference_mode():
        for start in tqdm(range(0, len(texts), batch_size)):
            batch_texts = texts[start:start + batch_size]

            encoded = tokenizer(
                batch_texts,
                padding=True,
                truncation=True,
                max_length=max_length,
                return_tensors="pt"
            ).to(device)

            outputs = model(**encoded)
            probabilities = torch.softmax(outputs.logits, dim=-1)

            all_probabilities.append(probabilities.cpu().numpy())

    probabilities = np.concatenate(all_probabilities, axis=0)
    predictions = probabilities.argmax(axis=1)

    results = dataframe.copy()
    results["predicted_label"] = predictions
    results["legitimate_probability"] = probabilities[:, 0]
    results["scam_probability"] = probabilities[:, 1]

    return results


v31_predictions = run_inference(
    model, tokenizer, v31_eval_df
)

llm_predictions = run_inference(
    model, tokenizer, llm_eval_df
)

print("Inference completed.")
print("V3.1 predictions:", v31_predictions.shape)
print("LLM-authored predictions:", llm_predictions.shape)

print("\nV3.1 prediction distribution:")
print(v31_predictions["predicted_label"].value_counts().sort_index())

print("\nLLM-authored prediction distribution:")
print(llm_predictions["predicted_label"].value_counts().sort_index())

  0%|          | 0/220 [00:00<?, ?it/s]

  0%|          | 0/204 [00:00<?, ?it/s]

Inference completed.
V3.1 predictions: (7013, 6)
LLM-authored predictions: (6506, 6)

V3.1 prediction distribution:
predicted_label
0    6227
1     786
Name: count, dtype: int64

LLM-authored prediction distribution:
predicted_label
0    6230
1     276
Name: count, dtype: int64


In [17]:
# Evaluate the existing DistilBERT model on both synthetic scam distributions

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    average_precision_score
)

def evaluate_predictions(results, dataset_name):
    y_true = results["label"].astype(int)
    y_pred = results["predicted_label"].astype(int)
    scam_probabilities = results["scam_probability"]

    print(f"\n{'=' * 75}")
    print(dataset_name)
    print("=" * 75)

    print("Total records:", len(results))
    print("Actual scams:", int(y_true.sum()))
    print("Actual legitimate:", int((y_true == 0).sum()))
    print("Predicted scams:", int(y_pred.sum()))

    print("\nClassification metrics:")
    print(f"Accuracy:       {accuracy_score(y_true, y_pred):.6f}")
    print(f"Scam Precision: {precision_score(y_true, y_pred, zero_division=0):.6f}")
    print(f"Scam Recall:    {recall_score(y_true, y_pred, zero_division=0):.6f}")
    print(f"Scam F1:        {f1_score(y_true, y_pred, zero_division=0):.6f}")
    print(f"Macro F1:       {f1_score(y_true, y_pred, average='macro', zero_division=0):.6f}")
    print(f"PR-AUC:         {average_precision_score(y_true, scam_probabilities):.6f}")

    print("\nConfusion Matrix:")
    print("Rows = Actual, Columns = Predicted")
    print("Labels = [LEGITIMATE, SCAM]")
    print(confusion_matrix(y_true, y_pred, labels=[0, 1]))

    print("\nClassification Report:")
    print(classification_report(
        y_true,
        y_pred,
        labels=[0, 1],
        target_names=["LEGITIMATE", "SCAM"],
        digits=4,
        zero_division=0
    ))


evaluate_predictions(v31_predictions, "V3.1 Synthetic Evaluation")
evaluate_predictions(llm_predictions, "LLM-authored Synthetic Evaluation")


V3.1 Synthetic Evaluation
Total records: 7013
Actual scams: 780
Actual legitimate: 6233
Predicted scams: 786

Classification metrics:
Accuracy:       0.998859
Scam Precision: 0.991094
Scam Recall:    0.998718
Scam F1:        0.994891
Macro F1:       0.997125
PR-AUC:         0.999443

Confusion Matrix:
Rows = Actual, Columns = Predicted
Labels = [LEGITIMATE, SCAM]
[[6226    7]
 [   1  779]]

Classification Report:
              precision    recall  f1-score   support

  LEGITIMATE     0.9998    0.9989    0.9994      6233
        SCAM     0.9911    0.9987    0.9949       780

    accuracy                         0.9989      7013
   macro avg     0.9955    0.9988    0.9971      7013
weighted avg     0.9989    0.9989    0.9989      7013


LLM-authored Synthetic Evaluation
Total records: 6506
Actual scams: 273
Actual legitimate: 6233
Predicted scams: 276

Classification metrics:
Accuracy:       0.998309
Scam Precision: 0.974638
Scam Recall:    0.985348
Scam F1:        0.979964
Macro F1:   

In [18]:
# Inspect synthetic scam messages incorrectly classified as legitimate

false_negatives = pd.concat(
    [
        v31_predictions.loc[
            (v31_predictions["label"] == 1) &
            (v31_predictions["predicted_label"] == 0)
        ],
        llm_predictions.loc[
            (llm_predictions["label"] == 1) &
            (llm_predictions["predicted_label"] == 0)
        ]
    ],
    ignore_index=True
).copy()

false_negatives["token_count"] = false_negatives["normalized_text"].apply(
    lambda text: len(
        tokenizer(
            str(text),
            truncation=False,
            add_special_tokens=True
        )["input_ids"]
    )
)

false_negatives = false_negatives.sort_values(
    "scam_probability"
).reset_index(drop=True)

print("Total false negatives:", len(false_negatives))

for index, row in false_negatives.iterrows():
    print(f"\n{'=' * 90}")
    print(f"False Negative {index + 1}")
    print("=" * 90)
    print("Source:", row["source"])
    print("Actual label:", row["label"])
    print("Predicted label:", row["predicted_label"])
    print(f"Scam probability: {row['scam_probability']:.6f}")
    print(f"Legitimate probability: {row['legitimate_probability']:.6f}")
    print("Token count:", row["token_count"])
    print("Exceeds 512 tokens:", row["token_count"] > 512)
    print("\nMessage:")
    print(row["normalized_text"])

Total false negatives: 5

False Negative 1
Source: LLM-authored
Actual label: 1
Predicted label: 0
Scam probability: 0.001919
Legitimate probability: 0.998081
Token count: 77
Exceeds 512 tokens: False

Message:
hey buddy you seeing the flash floods down in the valley right now?? whole towns are underwater and thousands of families are stranded with zero dry food or drinking water. a bunch of us set up an emergency supplies pool through Solaria Aid. even pitching in 15 or 20 bucks gets emergency ration packs delivered by boat tonight. tap here to chip in: [URL]

False Negative 2
Source: V3.1 Synthetic
Actual label: 1
Predicted label: 0
Scam probability: 0.001932
Legitimate probability: 0.998068
Token count: 141
Exceeds 512 tokens: False

Message:
[11:20 AM] Talent_Coordinator_Alex: E-Commerce Role: Part-time product cataloging assistant position with flexible hours and weekly payments. [11:22 AM] Applicant: Is prior experience required, or is training provided? [11:23 AM] Talent_Coordin